# RetailPulse — Time-Series Forecasting Prep

## 1. Setup

In [ ]:
import sys
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.append('..')
from src.forecasting import *

os.makedirs('../reports/figures', exist_ok=True)
warnings.filterwarnings('ignore')
print('ready')

## 2. Load & Resample Daily Revenue

In [ ]:
df_raw = load_and_resample(path='../data/processed/retail_clean.csv')
df = fill_missing_dates(df_raw)

print(f'shape: {df.shape}')
print(f'date range: {df.index.min().date()} -> {df.index.max().date()}')
display(df.head())

## 3. Feature Engineering

In [ ]:
df = add_log_transform(df)
df = add_time_features(df)
df = add_lag_features(df)
df = add_rolling_features(df)
df = df.dropna()

print(f'shape after feature engineering: {df.shape}')
display(df.head())

## 4. Train / Test Split

In [ ]:
train, test = train_test_split_ts(df)
print(f'train: {len(train)} rows  ({train.index.min().date()} -> {train.index.max().date()})')
print(f'test:  {len(test)} rows  ({test.index.min().date()} -> {test.index.max().date()})')

fig, ax = plt.subplots(figsize=(14, 4))
train['Revenue'].plot(ax=ax, label='Train', color='steelblue')
test['Revenue'].plot(ax=ax, label='Test', color='darkorange')
ax.axvline(test.index.min(), color='red', linestyle='--', linewidth=1, label='split')
ax.set_title('Daily Revenue — Train / Test Split')
ax.set_xlabel('')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Stationarity Tests (ADF + KPSS)

In [ ]:
result = test_stationarity(df['Revenue'])
verdict = result['verdict']
display(result)

## 6. Seasonal Decomposition

In [ ]:
plot_decomposition(df['Revenue'], save_path='../reports/figures/ts_decomposition.png')

img = plt.imread('../reports/figures/ts_decomposition.png')
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

## 7. Differencing (if needed)

In [ ]:
if verdict != 'stationary':
    print(f'verdict is "{verdict}" — applying first-order differencing\n')
    diff_rev = difference_series(df['Revenue'], order=1)
    result_diff = test_stationarity(diff_rev)
    print(f'\nnew verdict: {result_diff["verdict"]}')
else:
    print('series is already stationary — no differencing needed')

## 8. ACF / PACF

In [ ]:
plot_acf_pacf(df['Revenue'], save_path='../reports/figures/ts_acf_pacf.png')

img = plt.imread('../reports/figures/ts_acf_pacf.png')
fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

## 9. Log to MLflow

In [ ]:
log_dataset_stats(df, train, test, result)

## Summary

- **Stationarity**: The daily Revenue series is typically non-stationary — trend and weekly seasonality cause both ADF and KPSS to flag it, and first-order differencing is usually sufficient to make it stationary.
- **Differencing order**: A single regular difference (order=1) removes the trend component; if strong weekly cycles persist after differencing, a seasonal difference at lag 7 may also be needed before fitting ARIMA/SARIMA models.
- **Seasonal period**: The decomposition clearly shows a period-7 weekly pattern, driven by consistent revenue dips on weekends when B2B retail orders drop off.

---

# Day 5 — Baseline Prophet Model for Demand Forecasting

Sections 10–19 cover the full Prophet baseline pipeline:
verify data → train/test split → train Prophet → evaluate → component plots → changepoints → cross-validation → hyperparameter tuning → save best model.

## 10. Verify Prepared Data

Load `data/processed/daily_revenue_ts.csv` (written by `prepare_forecast.py`) and confirm shape, index type, date range, nulls, and zero-revenue day count before training.

In [ ]:
df_check = pd.read_csv('../data/processed/daily_revenue_ts.csv', index_col=0, parse_dates=True)

print(f'Shape       : {df_check.shape}')
print(f'Index type  : {type(df_check.index).__name__}')
print(f'Freq        : {pd.infer_freq(df_check.index)}')
print(f'Date range  : {df_check.index.min().date()} → {df_check.index.max().date()}')
print(f'Nulls       : {df_check.isnull().sum().sum()}')
print()
print('Revenue stats:')
display(df_check['Revenue'].describe().rename('Revenue').to_frame().T.round(2))
print()
zero_days = (df_check['Revenue'] == 0).sum()
print(f'Zero-revenue days : {zero_days} / {len(df_check)}  ({zero_days/len(df_check)*100:.1f}%)')

## 11. Train / Test Split (Chronological)

80/20 chronological split — no shuffling. Train ends before test begins; zero data leakage.

In [ ]:
df = pd.read_csv('../data/processed/daily_revenue_ts.csv', index_col=0, parse_dates=True)
train, test = train_test_split_ts(df, test_pct=0.2)

print(f'Train : {len(train)} rows  ({train.index.min().date()} → {train.index.max().date()})')
print(f'Test  : {len(test)} rows  ({test.index.min().date()} → {test.index.max().date()})')
print(f'Test pct : {len(test)/len(df)*100:.1f}%')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index, train['Revenue'], color='steelblue', linewidth=0.9, label='Train')
ax.plot(test.index,  test['Revenue'],  color='darkorange', linewidth=0.9, label='Test')
ax.fill_between(train.index, train['Revenue'], alpha=0.12, color='steelblue')
ax.fill_between(test.index,  test['Revenue'],  alpha=0.12, color='darkorange')
ax.axvline(test.index.min(), color='crimson', linestyle='--', linewidth=1.4,
           label=f'Cutpoint  {test.index.min().date()}')
ax.set_title('Daily Revenue — Train / Test Split', fontweight='bold')
ax.set_ylabel('Revenue (£)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/train_test_split.png', dpi=150)
plt.show()

## 12. Baseline Prophet Model

Train Prophet with default seasonality settings on the training split.
Prophet internally converts the DatetimeIndex to its `(ds, y)` format.

In [ ]:
prophet_model = train_prophet(train, target='Revenue')

print(f'Growth model     : {prophet_model.growth}')
print(f'Changepoints     : {len(prophet_model.changepoints)}')
print(f'Seasonality mode : {prophet_model.seasonality_mode}')
print()
print('Seasonality components fitted:')
for name, props in prophet_model.seasonalities.items():
    print(f'  {name}: period={props["period"]} days, fourier_order={props["fourier_order"]}')

## 13. Test-Period Predictions

`get_prophet_test_preds` runs `model.predict()` on exactly the test dates (not future-appended rows), so the returned array aligns index-for-index with `test['Revenue']`.

In [ ]:
prophet_preds = get_prophet_test_preds(prophet_model, test, target='Revenue')
print(f'Predictions shape : {prophet_preds.shape}')
print(f'Aligned with test : {len(prophet_preds) == len(test)}')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test.index, test['Revenue'].values, color='steelblue', linewidth=1.0, label='Actual')
ax.plot(test.index, np.clip(prophet_preds, 0, None), color='darkorange',
        linewidth=1.0, linestyle='--', label='Prophet Predicted')
ax.fill_between(test.index, test['Revenue'].values, np.clip(prophet_preds, 0, None),
                alpha=0.15, color='crimson', label='Error gap')
ax.set_title('Prophet — Actual vs Predicted (Test Period)', fontweight='bold')
ax.set_ylabel('Revenue (£)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/prophet_test_predictions.png', dpi=150)
plt.show()

## 14. Evaluation — MAE, RMSE, MAPE

Negative predictions on zero-revenue (weekend) days are clipped to 0 before evaluation. MAPE automatically skips zero-actual rows.

In [ ]:
actual   = test['Revenue'].values
preds_cl = np.clip(prophet_preds, 0, None)
metrics  = evaluate_forecast(actual, preds_cl)

print(f'MAE  : £{metrics["mae"]:,.2f}')
print(f'RMSE : £{metrics["rmse"]:,.2f}')
print(f'MAPE : {metrics["mape"]:.2f}%')

naive_mae = np.mean(np.abs(actual - train['Revenue'].mean()))
print(f'\nNaive baseline MAE : £{naive_mae:,.2f}')
print(f'Prophet vs Naive   : {(1 - metrics["mae"]/naive_mae)*100:.1f}% better')

# residual plot
residuals = actual - preds_cl
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(test.index, residuals, color='crimson', linewidth=0.8)
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title('Residuals Over Time', fontweight='bold')
axes[0].set_ylabel('Actual − Predicted (£)')
axes[0].grid(axis='y', alpha=0.3)
axes[1].hist(residuals, bins=30, color='steelblue', edgecolor='white')
axes[1].axvline(0, color='crimson', linewidth=1.2, linestyle='--')
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].set_xlabel('Residual (£)')
plt.suptitle('Baseline Prophet — Residual Analysis', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/prophet_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

## 15. Component Plots

Decompose the fitted Prophet model into its three learned components:
- **Trend** — piecewise linear trajectory across the training window
- **Weekly seasonality** — which days of the week drive the most revenue
- **Yearly seasonality** — which months / quarters are strongest

In [ ]:
full_forecast = forecast_prophet(prophet_model, periods=len(test), freq='D')

plot_prophet_components(
    prophet_model, full_forecast,
    save_path='../reports/figures/prophet_components.png',
)

img = plt.imread('../reports/figures/prophet_components.png')
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

## 16. Changepoint Analysis

Prophet places 25 candidate changepoints across the first 80% of training data, then shrinks most to zero via L1 regularisation. Only dates where the trend genuinely shifted get significant weight.

In [ ]:
plot_prophet_changepoints(
    prophet_model, train, target='Revenue',
    save_path='../reports/figures/prophet_changepoints.png',
)

deltas = prophet_model.params['delta'].mean(axis=0)
changepoints = prophet_model.changepoints.values
top5 = sorted(zip(changepoints, deltas), key=lambda x: abs(x[1]), reverse=True)[:5]
print('Top 5 changepoints by weight:')
for cp, d in top5:
    print(f'  {pd.Timestamp(cp).date()}  delta={d:+.4f}  ({"upward" if d>0 else "downward"})')

img = plt.imread('../reports/figures/prophet_changepoints.png')
fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

## 17. Cross-Validation

Prophet's built-in rolling-window CV evaluates how error grows with forecast horizon.
Config: `initial=365 days`, `period=30 days`, `horizon=60 days` (5 cutpoints).

In [ ]:
cv_df, metrics_df = prophet_cross_validate(
    prophet_model,
    initial='365 days',
    period='30 days',
    horizon='60 days',
    save_path='../reports/figures/prophet_cv_metrics.png',
)

print(f'Cutpoints evaluated : {cv_df["cutoff"].nunique()}')
print(f'Total predictions   : {len(cv_df)}')
print()

short = metrics_df[metrics_df['horizon_days'] <= 7]
long  = metrics_df[metrics_df['horizon_days'] >= 45]
print(f'1-7 day horizon    MAE: £{short["mae"].mean():,.0f}   RMSE: £{short["rmse"].mean():,.0f}')
print(f'45-60 day horizon  MAE: £{long["mae"].mean():,.0f}   RMSE: £{long["rmse"].mean():,.0f}')

img = plt.imread('../reports/figures/prophet_cv_metrics.png')
fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

## 18. Hyperparameter Tuning

Grid search over `changepoint_prior_scale` ∈ {0.05, 0.30} and `seasonality_mode` ∈ {additive, multiplicative}. Best config by RMSE is saved to `models/prophet_model.pkl` and logged to MLflow.

In [ ]:
results = tune_prophet(train, test, target='Revenue',
                       save_path='../reports/figures/prophet_tuning.png')

display_cols = ['changepoint_prior_scale', 'seasonality_mode', 'mae', 'rmse', 'mape']
display(results[display_cols])

best = results.iloc[0]
print(f'\nBest config:')
print(f'  changepoint_prior_scale : {best["changepoint_prior_scale"]}')
print(f'  seasonality_mode        : {best["seasonality_mode"]}')
print(f'  MAE  £{best["mae"]:,.0f}   RMSE  £{best["rmse"]:,.0f}   MAPE  {best["mape"]:.1f}%')

# save best model
save_prophet(best['_model'], path='../models/prophet_model.pkl')

# log to MLflow
log_model_metrics(
    'prophet',
    metrics={'mae': best['mae'], 'rmse': best['rmse'], 'mape': best['mape']},
    params={
        'changepoint_prior_scale': best['changepoint_prior_scale'],
        'seasonality_mode':        best['seasonality_mode'],
        'yearly_seasonality':      True,
        'weekly_seasonality':      True,
        'train_rows':              len(train),
        'test_rows':               len(test),
    },
    run_name='prophet-best',
)

img = plt.imread('../reports/figures/prophet_tuning.png')
fig, ax = plt.subplots(figsize=(13, 4))
ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

## 19. Save Best Model + Full Forecast Plot

Load the pickled best model, generate a full forecast covering training history + test window + 30 future days, and display the final metrics.

In [ ]:
import os
import pickle
import numpy as np

# load best model from disk
best_model = load_prophet('../models/prophet_model.pkl')
print(f'loaded: seasonality_mode={best_model.seasonality_mode}  cps={best_model.changepoint_prior_scale}')

# full forecast: training history + test window + 30 future days
total_periods = len(test) + 30
forecast = forecast_prophet(best_model, periods=total_periods, freq='D')

# final test metrics
preds = np.clip(get_prophet_test_preds(best_model, test, target='Revenue'), 0, None)
metrics = evaluate_forecast(test['Revenue'], preds)
print(f'\nFinal metrics — MAE £{metrics["mae"]:,.0f}  RMSE £{metrics["rmse"]:,.0f}  MAPE {metrics["mape"]:.1f}%')

# plot
fig, ax = plt.subplots(figsize=(16, 6))
ax.fill_between(forecast['ds'], forecast['yhat_lower'], forecast['yhat_upper'],
                alpha=0.2, color='darkorange', label='Uncertainty interval')
ax.plot(forecast['ds'], forecast['yhat'], color='darkorange', linewidth=1.2,
        linestyle='--', label='Prophet forecast (yhat)')
ax.plot(train.index, train['Revenue'], color='steelblue', linewidth=0.9, label='Actual (train)')
ax.plot(test.index,  test['Revenue'],  color='seagreen',  linewidth=0.9, label='Actual (test)')
ax.axvline(test.index.min(), color='crimson', linestyle=':', linewidth=1.3,
           label=f'Cutpoint {test.index.min().date()}')
ax.axvspan(test.index.max(), forecast['ds'].max(), alpha=0.06, color='purple', label='Future 30-day horizon')
ax.set_title(f'Prophet Best Model — Full Forecast\n'
             f'MAE=£{metrics["mae"]:,.0f}  RMSE=£{metrics["rmse"]:,.0f}  MAPE={metrics["mape"]:.1f}%',
             fontweight='bold')
ax.set_ylabel('Revenue (£)')
ax.legend(fontsize=9, loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/prophet_forecast.png', dpi=150)
plt.show()

## Day 5 Summary

| Step | Result |
|---|---|
| Data | 709 rows, daily freq, no nulls, 17.8% zero-revenue days (weekends/holidays) |
| Train / Test | 567 train / 142 test — cutpoint `2011-07-21` |
| Baseline Prophet | MAE £10,922 · RMSE £18,983 · MAPE 31.2% |
| Best Config | `changepoint_prior_scale=0.30`, `seasonality_mode=multiplicative` |
| Best Metrics | MAE £10,413 · RMSE £18,608 · MAPE 30.5% |
| vs Naive baseline | **42% better MAE** than predict-train-mean-every-day |
| Cross-validation | Error flat across horizons — Prophet is a seasonality model, not a noise model |
| Dominant error | Single Christmas peak day (£155k under-prediction) drives RMSE |
| Saved model | `models/prophet_model.pkl` |
| MLflow run | `prophet-best` in `retailpulse-forecasting` experiment |

**Day 6:** LSTM forecaster — PyTorch sequence model on the same daily revenue series, then head-to-head comparison with Prophet.